# Creating agentic workflows in LlamaIndex

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
!uv add llama-index-utils-workflow

Resolved 399 packages in 15ms
Audited 370 packages in 58ms


## Creating a Basic Step and Workflow

In [6]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step

class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        ...
        return StopEvent(result="Hello, world!")


w = MyWorkflow(timeout=10, verbose=True)
result = await w.run()
result

[tick] add: StartEvent()
[my_step:0] started from StartEvent
[result] StopEvent(result='Hello, world!')
[my_step:0] complete with StopEvent


'Hello, world!'

## Connecting Multiple Steps

In [8]:
from llama_index.core.workflow import Event

class ProcessingEvent(Event):
    intermediate_result: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        ...
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(timeout=10, verbose=True)
result = await w.run()
result

[tick] add: StartEvent()
[step_one:0] started from StartEvent
[step_one:0] complete with ProcessingEvent
[tick] add: ProcessingEvent(intermediate_result='Step 1 complete')
[step_two:0] started from ProcessingEvent
[result] StopEvent(result='Finished processing: Step 1 complete')
[step_two:0] complete with StopEvent


'Finished processing: Step 1 complete'

## Loops and Branches

In [10]:
from llama_index.core.workflow import Event
import random


class ProcessingEvent(Event):
    intermediate_result: str


class LoopEvent(Event):
    loop_output: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(verbose=False)
result = await w.run()
result

Bad thing happened
Bad thing happened
Good thing happened


'Finished processing: First step complete.'

## State Management

In [11]:
from llama_index.core.workflow import Context, StartEvent, StopEvent, Workflow, step


class QueryWorkflow(Workflow):
    @step
    async def query(self, ctx: Context, ev: StartEvent) -> StopEvent:
        # store query in the context
        await ctx.store.set("query", "What is the capital of France?")

        # do something with context and event
        val = "Paris"

        # retrieve query from the context
        query = await ctx.store.get("query")
        print(f"Stored query: {query}")

        return StopEvent(result=val)


w = QueryWorkflow(timeout=10, verbose=False)
result = await w.run()
result

Stored query: What is the capital of France?


'Paris'

## Automating workflows with Multi-Agent Workflows


In [12]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
# from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.llms.openai import OpenAI



def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

llm = OpenAI(model="gpt-5-nano")


multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

# Create the workflow
workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
)

agent_output = await workflow.run(user_msg="Can you add 51 and 3?")
agent_output.response.blocks[0].text


'54'

## Handling State in Multi-Agent Workflows



In [15]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.core.workflow import Context


async def add(ctx: Context, a: int, b: int) -> int:
    """Add two numbers."""
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)

    return a + b


async def multiply(ctx: Context, a: int, b: int) -> int:
    """Multiply two numbers."""
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)

    return a * b


multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="add_agent",
    initial_state={"num_fn_calls": 0},
)

ctx = Context(workflow)
response = await workflow.run(user_msg="Can you add 5 and 3, then multiply the result by 2?", ctx=ctx)

state = await ctx.store.get("state")
print(state["num_fn_calls"])
print(response)

1
16
